# Logistic Regression — A Beginner's Walkthrough
### From raw data to a trained, evaluated, tuned model — one small step at a time

This notebook is meant to be run **one cell at a time, live in class**. Every piece of code is preceded by *why* we're doing it, and followed by *what the output means*. Nothing here is hidden inside a big function — you will see every step happen.

---
## 1. What are we actually trying to do?
---

Before any code: three questions, always, for every ML problem.

**What is one row?** One customer.

**What are the features?** The information we already know about that customer — Age, Income, City, Education.

**What is the target?** The thing we want to predict — did they buy the product? (`Bought` = 0 or 1)

$$
\text{Age} + \text{Income} + \text{Education} \;\longrightarrow\; \text{Logistic Regression} \;\longrightarrow\; \text{Probability} \;\longrightarrow\; 0 \text{ or } 1
$$

That's the entire notebook, in one line. Everything below is just carefully unpacking each arrow.

## 2. Our dataset

A small dataset on purpose — small enough that you can look at every single row and understand it.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Age":       [22, 25, 47, 52, 46, 56, 33, 38, 41, 29, 61, 27, 44, 35, 58, 31, 49, 39],
    "Income":    [20000, 25000, 60000, 58000, 62000, 70000, 40000, 45000, 50000, 32000,
                  75000, 28000, 55000, 42000, 68000, 30000, 61000, 47000],
    "City":      ["Chennai","Mumbai","Delhi","Chennai","Mumbai","Delhi","Chennai","Mumbai",
                  "Delhi","Chennai","Mumbai","Delhi","Chennai","Mumbai","Delhi","Chennai","Mumbai","Delhi"],
    "Education": ["School","School","Graduate","Postgraduate","Graduate","Postgraduate","Diploma","Diploma",
                  "Graduate","School","Postgraduate","Diploma","Graduate","Diploma","Postgraduate","School","Graduate","Diploma"],
    "Bought":    [0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}
df = pd.DataFrame(data)
df


,Age,Income,City,Education,Bought
0,22,20000,Chennai,School,0
1,25,25000,Mumbai,School,0
2,47,60000,Delhi,Graduate,1
3,52,58000,Chennai,Postgraduate,1
4,46,62000,Mumbai,Graduate,1
5,56,70000,Delhi,Postgraduate,1
6,33,40000,Chennai,Diploma,0
7,38,45000,Mumbai,Diploma,0
8,41,50000,Delhi,Graduate,1
9,29,32000,Chennai,School,0


**Reading this table:** each row is one customer. `Age`, `Income`, `City`, `Education` are things we know *before* they decide to buy. `Bought` is what actually happened — 1 if they bought, 0 if they didn't. That last column is the one thing we won't know in advance for a *new* customer — it's what we're building a model to predict.

In [ ]:
df["Bought"].value_counts()


,count
Bought,
0,9
1,9


9 customers bought, 9 didn't — a nice, balanced dataset to learn on.

**Features vs. target, in words:** `Age`, `Income`, `City`, `Education` are our **features** (what we already know). `Bought` is our **target** (what we want to predict). We'll build the actual numeric feature table in a moment — first, an important ordering decision.

---
## 3. Train-Test Split — done FIRST, before any encoding or scaling
---

$$
\text{Original Data} \;\longrightarrow\; \text{Training Data} + \text{Test Data}
$$

**Why we split before touching the data any further:** every single preprocessing step that *learns* something from the data — encoding categories, computing a mean and standard deviation for scaling — must only ever learn from the training rows. If we let any of these steps see the test rows first, information quietly leaks from data that's supposed to be completely unseen. The safest way to guarantee this is to split the raw data apart *first*, before any encoding or scaling exists at all.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.30,      # 30% of rows held out for testing
    random_state=42,     # makes the split reproducible -- same split every time we run this
    stratify=df["Bought"]   # keeps the 0/1 ratio the same in both train and test
)

print("Train rows:", len(df_train), " Test rows:", len(df_test))


Train rows: 12  Test rows: 6


In [ ]:
df_train

,Age,Income,City,Education,Bought
10,61,75000,Mumbai,Postgraduate,1
11,27,28000,Delhi,Diploma,0
5,56,70000,Delhi,Postgraduate,1
2,47,60000,Delhi,Graduate,1
15,31,30000,Chennai,School,0
14,58,68000,Delhi,Postgraduate,1
0,22,20000,Chennai,School,0
8,41,50000,Delhi,Graduate,1
6,33,40000,Chennai,Diploma,0
3,52,58000,Chennai,Postgraduate,1


**`test_size=0.30`** — 30% held back for testing, 70% for training.
**`random_state=42`** — a fixed seed so the split is identical every time we re-run this cell.
**`stratify=df["Bought"]`** — keeps roughly the same proportion of 0s and 1s in both `df_train` and `df_test`.

Everything from here — encoding, scaling, the model itself — will only ever be *fit* on `df_train`.

---
## 4. Turning categories into numbers
---

`City` and `Education` are text, but our model only understands numbers. We need to **encode** them — but not the same way, because they're fundamentally different kinds of categories.

### Nominal vs. Ordinal

**Nominal** — no natural order.
> City: Chennai, Mumbai, Delhi. None of these is "more" or "less" than another.

**Ordinal** — a real, natural order exists.
> Education: School < Diploma < Graduate < Postgraduate. There IS a meaningful order here.

### 4a. One-Hot Encoding — for nominal categories (`City`)

**Important:** if we just gave City the numbers 0, 1, 2, the model would think Delhi is somehow "more" than Chennai — a fake order that doesn't exist. One-Hot Encoding avoids this by creating a separate 0/1 column *per category* instead.

**And just like scaling: this encoder must only *learn* its categories from the training data.**

In [ ]:
from sklearn.preprocessing import OneHotEncoder

city_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Learn which categories exist -- from TRAINING data only
city_encoder.fit(df_train[["City"]])

print("Categories learned:", city_encoder.categories_)


Categories learned: [array(['Chennai', 'Delhi', 'Mumbai'], dtype=object)]


**`handle_unknown="ignore"`** — worth pausing on. What if the *test* set contained a city the training set never saw (say, "Bangalore")? Because the encoder only learned from `df_train`, it has no column for a city it's never seen. `handle_unknown="ignore"` tells it: encode that row as all-zeros across the known City columns, rather than crashing. This is a genuinely common real-world situation, and precisely why we `.fit()` only on training data — it forces us to handle this case properly, instead of silently hiding it (which is exactly what would happen if we'd encoded the *entire* dataset before splitting).

In [ ]:
city_train = city_encoder.transform(df_train[["City"]])
city_test = city_encoder.transform(df_test[["City"]])   # reuses the SAME learned categories -- never re-fit

print("City columns:", city_encoder.get_feature_names_out())
print("city_train shape:", city_train.shape, " city_test shape:", city_test.shape)
city_train



City columns: ['City_Chennai' 'City_Delhi' 'City_Mumbai']
city_train shape: (12, 3)  city_test shape: (6, 3)


array([[0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler()
mms_fit = mms.fit(df_train['Age'].values.reshape(-1, 1))
mms_transform = mms.transform(df_train['Age'].values.reshape(-1, 1))
mms_transform


mms_fit_trans = mms.fit_transform(df_train['Age'].values.reshape(-1, 1))
mms_fit_trans

array([[1.        ],
       [0.12820513],
       [0.87179487],
       [0.64102564],
       [0.23076923],
       [0.92307692],
       [0.        ],
       [0.48717949],
       [0.28205128],
       [0.76923077],
       [0.07692308],
       [0.43589744]])

### 4b. Ordinal Encoding — for ordered categories (`Education`)

**Common beginner mistake:** Python does **not** automatically know that "Postgraduate" should come after "Graduate." Left to its own defaults, it might alphabetize them — destroying the real order. We tell it the order explicitly, and once again, `.fit()` only on the training rows.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

education_order = ["School", "Diploma", "Graduate", "Postgraduate"]

education_encoder = OrdinalEncoder(categories=[education_order])
education_encoder.fit(df_train[["Education"]])   # learn (here: just confirm) the order, from TRAINING data only

edu_train = education_encoder.transform(df_train[["Education"]])
edu_test = education_encoder.transform(df_test[["Education"]])

print("First 5 training rows, Education encoded:", edu_train[:5].ravel())
edu_train[:-1]


First 5 training rows, Education encoded: [3. 1. 3. 2. 0.]


array([[3.],
       [1.],
       [3.],
       [2.],
       [0.],
       [3.],
       [0.],
       [2.],
       [1.],
       [3.],
       [0.]])

**Reading the output:** School → 0, Diploma → 1, Graduate → 2, Postgraduate → 3 — exactly the real order, because we told `OrdinalEncoder` what that order was via `categories=`.

**Note on this specific encoder:** since we *fixed* the category order ourselves (rather than letting the encoder discover it from data), fitting on train vs. the full dataset wouldn't change the learned mapping here. We still fit only on `df_train` regardless — it keeps the exact same discipline as every other step, and it's the pattern that *does* matter the moment an encoder is learning something genuinely data-dependent (like `OneHotEncoder`'s category list above, or `StandardScaler`'s mean and std coming up next).

### 4c. Important — Encoding is NOT the same as Weighting

**Common beginner mistake:** seeing `Postgraduate = 3` and assuming this means "3 times more important" than `School = 0`.

**It does not.** Encoding is only a *label* — a way to represent a category as a number so the model can use it. It carries no built-in statement about importance.

> If you specifically want to make one class matter more during training — for example, if `Bought=1` were rare and you wanted the model to pay extra attention to it — that's a completely separate idea, called **class weighting**:
> ```python
> LogisticRegression(class_weight="balanced")
> ```
> Don't confuse encoding a category with weighting a class. They solve two different problems.

### Assembling the final X_train and X_test

Combine the numeric columns (Age, Income) with the two sets of encoded columns — separately for train and test, since each piece above was already correctly split:

In [ ]:
X_train = np.hstack([
    df_train[["Age", "Income"]].values,
    edu_train,
    city_train
])
X_test = np.hstack([
    df_test[["Age", "Income"]].values,
    edu_test,
    city_test
])

y_train = df_train["Bought"].values
y_test = df_test["Bought"].values

print("X_train shape:", X_train.shape, " X_test shape:", X_test.shape)


X_train shape: (12, 6)  X_test shape: (6, 6)


---
## 5. Scaling
---

Look at `Age` (values like 22–61) versus `Income` (values like 20,000–75,000). Income's raw numbers are thousands of times larger — a model can be misled into thinking Income matters more, purely because of its units, not its actual importance. Scaling puts every feature on the same footing.

**The formula — standardization (z-score):**

$$
z = \frac{x - \mu}{\sigma}
$$

Subtract the mean ($\mu$), divide by the standard deviation ($\sigma$). The result: every feature has mean 0 and standard deviation 1.

### `.fit()` vs. `.transform()` vs. `.fit_transform()` — the exact difference

**`.fit()`** — *learn* something from the data (here: learn the mean $\mu$ and standard deviation $\sigma$).

**`.transform()`** — *use* what was already learned to convert data.

**`.fit_transform()`** — learn AND convert, in one step, on the same data.

This is the exact same pattern we just used twice for encoding — `StandardScaler` is no different.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Learn mean & std from TRAINING data, and transform it in the same step
X_train_scaled = scaler.fit_transform(X_train)

# Re-use the SAME mean & std (already learned) to transform the test data
X_test_scaled = scaler.transform(X_test)


In [ ]:
print("Mean learned from training data:", scaler.mean_.round(1))


Mean learned from training data: [4.10000e+01 4.75833e+04 1.60000e+00 3.00000e-01 5.00000e-01 2.00000e-01]


**Important — the single most common beginner mistake with scaling:**

$$
\text{TRAINING DATA} \xrightarrow{\texttt{fit\_transform()}} \text{learn } \mu,\sigma \text{ AND transform}
$$

$$
\text{TEST DATA} \xrightarrow{\texttt{transform()}} \text{use the SAME training } \mu,\sigma
$$

**Never do this:**
```python
X_test_scaled = scaler.fit_transform(X_test)   # WRONG
```
This would make the scaler learn a brand new mean and standard deviation *from the test set itself* — the same leakage mistake we already avoided above for both encoders, now applied to scaling.

---
## 6. Logistic Regression
---

Recall the math you've already been taught:

$$
z = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p
$$

$$
P(y=1) = \frac{1}{1 + e^{-z}}
$$

$$
\text{Linear equation} \;\rightarrow\; \text{Sigmoid} \;\rightarrow\; \text{Probability} \;\rightarrow\; \text{Threshold} \;\rightarrow\; \text{Class}
$$

Everything from here is just that flow, in code.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)


LogisticRegression()

**`.fit()`** here means: find the $\beta_0, \beta_1, \ldots$ values that best separate the two classes, using the training data.

In [ ]:
y_pred = model.predict(X_test_scaled)
y_pred


array([0, 0, 1, 0, 1, 1])

In [ ]:
y_pred_proba = model.predict_proba(X_test_scaled)
y_pred_proba


array([[0.70459328, 0.29540672],
       [0.76272734, 0.23727266],
       [0.24765805, 0.75234195],
       [0.95566365, 0.04433635],
       [0.22345966, 0.77654034],
       [0.40591584, 0.59408416]])

**`.predict()` vs. `.predict_proba()` — the difference:**

`.predict()` gives the final 0/1 answer, already decided using a 0.5 cutoff internally.

`.predict_proba()` gives the actual probability behind that decision — two numbers per row, `[P(class=0), P(class=1)]`, always summing to 1.

---
## 7. Metrics — how good is the model, really?
---

$$
\text{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}
$$

$$
\text{Precision} = \frac{TP}{TP+FP}
$$

$$
\text{Recall} = \frac{TP}{TP+FN}
$$

$$
F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))


Accuracy:  1.0
Precision: 1.0
Recall:    1.0
F1 Score:  1.0


## 8. Confusion Matrix

The four raw counts every metric above is built from:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| **Actual 0** | TN | FP |
| **Actual 1** | FN | TP |

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm


array([[3, 0],
       [0, 3]])

**Reading this exact output:** the sklearn matrix follows the same layout as the table above — top-left is TN, top-right is FP, bottom-left is FN, bottom-right is TP. Match each number in the array to that table before moving on.

---
## 9. Cross-Validation
---

**Why do we need this?**

$$
\text{One train-test split} \;\rightarrow\; \text{could be lucky or unlucky} \;\rightarrow\; \text{performance may change} \;\rightarrow\; \text{Cross-Validation}
$$

Our one test set gave us one accuracy number. But what if that particular split just happened to be easy (or hard)? Cross-validation answers this by testing on *several* different splits and looking at the whole spread of results, not just one.

### 5-Fold Cross-Validation, visually

```
Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5

TEST   | TRAIN  | TRAIN  | TRAIN  | TRAIN
TRAIN  | TEST   | TRAIN  | TRAIN  | TRAIN
TRAIN  | TRAIN  | TEST   | TRAIN  | TRAIN
TRAIN  | TRAIN  | TRAIN  | TEST   | TRAIN
TRAIN  | TRAIN  | TRAIN  | TRAIN  | TEST
```

Every row is trained on 4/5 of the data and tested on the remaining 1/5 — and every single row of data gets to be "the test set" exactly once, across the 5 rounds.

$$
\text{CV Mean} = \frac{S_1+S_2+S_3+S_4+S_5}{5}
$$

We also look at the **standard deviation** across the 5 scores — a small spread means the model's performance is stable no matter which rows it's tested on; a large spread is a warning sign.

### Important — scaling must happen fresh, INSIDE each fold

**Common beginner mistake:** scaling the whole training set once, *before* cross-validation begins. That lets information about the validation fold's own values quietly leak into the mean/std used to scale it — the same leakage idea from Sections 4 and 5, now happening one level deeper, inside cross-validation itself.

The correct order, for every single fold:

$$
\text{CV TRAIN} \xrightarrow{\texttt{fit\_transform()}} \text{learn } \mu,\sigma \;\rightarrow\; \text{transform CV TRAIN}
$$

$$
\text{CV VALIDATION} \xrightarrow{\texttt{transform()}} \text{use that SAME } \mu,\sigma
$$

**A fair question: should we also redo the City/Education encoding inside every fold, the same way?** In principle, yes, for full rigor — but in practice this is a much smaller concern here: our encoders aren't learning continuous statistics the way `StandardScaler` learns a mean and standard deviation from every value in the fold. `OrdinalEncoder`'s order was fixed by us, not discovered from data, and `OneHotEncoder`'s categories rarely change between folds of the same training set. We'll encode once on the full training set and focus our fold-by-fold care on scaling, where the leakage risk is real and continuous. Keep this distinction in mind — it's a legitimate simplification here, not a rule that never matters.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
fold_scores = []

for fold_number, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):

    X_cv_train = X_train[train_idx]
    X_cv_val = X_train[val_idx]
    y_cv_train = y_train[train_idx]
    y_cv_val = y_train[val_idx]

    # Scaling learned FRESH on this fold's training rows only
    fold_scaler = StandardScaler()
    X_cv_train_scaled = fold_scaler.fit_transform(X_cv_train)
    X_cv_val_scaled = fold_scaler.transform(X_cv_val)

    fold_model = LogisticRegression()
    fold_model.fit(X_cv_train_scaled, y_cv_train)

    score = fold_model.score(X_cv_val_scaled, y_cv_val)
    fold_scores.append(score)

    print(f"Fold {fold_number}: validation accuracy = {score:.3f}")


Fold 1: validation accuracy = 0.750
Fold 2: validation accuracy = 1.000
Fold 3: validation accuracy = 1.000


In [ ]:
print("All fold scores:", fold_scores)
print("CV Mean:", round(np.mean(fold_scores), 4))
print("CV Std: ", round(np.std(fold_scores), 4))


All fold scores: [0.75, 1.0, 1.0]
CV Mean: 0.9167
CV Std:  0.1179


**Reading this:** the mean tells us roughly how well the model performs on new data. The standard deviation tells us how much that performance *wobbles* depending on which rows happen to be in the test fold.

> **A note on Pipeline:** everything we've done by hand so far — encoding fit only on train, scaling fit only on train, redone fresh inside every CV fold — can be automated by wrapping these steps inside a single `Pipeline` object. We've deliberately done it manually first so you can see every step happening. Once you're comfortable with what's really going on underneath, Pipeline becomes a convenient shortcut for exactly this same process — nothing more, nothing magic.

---
## 10. Hyperparameter Tuning
---

### Parameter vs. Hyperparameter — a crucial distinction

**Parameter** — learned automatically BY the model, from data.
> $\beta_0, \beta_1, \beta_2, \ldots$ — the model finds these itself during `.fit()`.

**Hyperparameter** — chosen by US, before training even starts.
> `C` — we decide this value; the model never learns it on its own.

### What `C` actually does

$$
C \downarrow \;\Rightarrow\; \text{stronger regularization (simpler model)}
$$

$$
C \uparrow \;\Rightarrow\; \text{weaker regularization (more flexible model)}
$$

We could try values of `C` one at a time by hand — but `GridSearchCV` automates exactly that search, trying every candidate value and picking whichever performs best under cross-validation.

### Why do we suddenly need Pipeline here?

`GridSearchCV` builds its own internal cross-validation folds — the same fold-by-fold splitting from Section 9. If we scale the *whole* training set once, by hand, before handing it to `GridSearchCV`, every internal fold's validation rows would already be scaled using a mean/std that included those very rows — quietly leaking information, exactly the mistake Section 9 was built to avoid.

**This is precisely the situation `Pipeline` exists for.** Bundling the scaler and the model together means `GridSearchCV` can call `.fit()` on each fold's raw training rows — the Pipeline automatically does `fit_transform()` on that fold's training portion and `transform()` on that fold's validation portion, replicating our manual loop exactly, automatically, for every fold and every candidate `C`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Bundle scaler + model together, so each CV fold scales freshly and correctly
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

# Parameter name is "model__C" -- "which step" __ "which parameter"
param_grid = {"model__C": [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(pipe, param_grid, cv=3, scoring="accuracy")
grid_search.fit(X_train, y_train)   # RAW (unscaled) data in -- the pipeline scales internally, per fold

print("Best C found:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)


Best C found: {'model__C': 1}
Best CV accuracy: 1.0


**Reading this:** `GridSearchCV` repeated our manual cross-validation loop above, once for every candidate `C` value — and because we handed it a `Pipeline` instead of a bare model, each of those repetitions scaled correctly, fold by fold, exactly like we did by hand.

### The last step — evaluating on the untouched test set

Everything so far — including every number `GridSearchCV` just reported — came only from the **training** data. `X_test`/`y_test` have not been touched since Section 3. This is deliberate: tuning decisions (which `C` to use) must never be influenced by the test set, or it stops being a fair, final check.

Now that the best `C` is chosen, we evaluate **once**, on the test set, to get the number that actually matters.

In [ ]:
# grid_search already re-fit the best pipeline on the FULL training set automatically (refit=True is the default)
best_model = grid_search.best_estimator_

test_accuracy = best_model.score(X_test, y_test)   # raw X_test in -- the pipeline scales it internally
print("Final test accuracy (on the untouched test set):", test_accuracy)


Final test accuracy (on the untouched test set): 1.0


**This is the very last box in our mental-model flowchart:** `Final Model -> Untouched Test Set -> Final Performance`. This one number — not the cross-validation score, not any training accuracy along the way — is the honest answer to "how good is this model."<br><br>**Important:** if this number looks disappointing, the fix is never to go back and re-tune using the test set — that would defeat its entire purpose. The test set gets looked at once, at the very end, and whatever it says is the real answer.

---
## 11. The Complete Mental Model
---

```
Problem
   |
Understand Data
   |
Train-Test Split   <-- done FIRST, before any encoding or scaling
   |
Encoding            (fit on train, transform on test)
   |
Scaling             (fit on train, transform on test)
   |
Model
   |
Prediction
   |
Metrics
   |
Cross-Validation    (re-fit scaling fresh inside every fold)
   |
Hyperparameter Tuning  (Pipeline needed for correct per-fold scaling)
   |
Final Model
   |
Untouched Test Set
   |
Final Performance
```

### Cheat sheet

| Concept | Meaning |
|---|---|
| X | Features |
| y | Target |
| fit | Learn |
| transform | Apply what was already learned |
| fit_transform | Learn + transform, in one step |
| predict | Final class prediction (0/1) |
| predict_proba | Probability behind that prediction |
| CV | Evaluate across multiple folds, not just one split |
| Test set | Final, untouched evaluation — never used for tuning |
| Parameter | Learned automatically by the model |
| Hyperparameter | Chosen by us, before training |
| Pipeline | Bundles preprocessing + model so fit/transform happens correctly, automatically, per fold |